In [1]:
# flat100-4-flat100-7.cnf-90-c
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.random import random_circuit
import numpy as np
import json
from qiskit.quantum_info import Operator
from qiskit.synthesis import gridsynth_unitary, gridsynth_rz
import phoenix

CLIFFORD_T = ["h", "s", "sdg", "t", "x", "cx"]

# ---------------------------------------------------------------------------
# (A) One-liner via the transpiler plugin
# ---------------------------------------------------------------------------
def to_clifford_t_transpile(qc: QuantumCircuit) -> QuantumCircuit:
    """Whole-circuit Clifford+T synthesis via the gridsynth UnitarySynthesis
    plugin.  Simplest, but the ``epsilon`` in ``unitary_synthesis_plugin_config``
    does NOT currently propagate through ``transpile`` (a fixed internal
    precision is used); use approach (B) when you need to control epsilon."""
    return transpile(
        qc,
        basis_gates=CLIFFORD_T,
        unitary_synthesis_method="gridsynth",
        unitary_synthesis_plugin_config={"epsilon": 1e-10},  # (currently ignored by transpile)
        optimization_level=1,
    )


# ---------------------------------------------------------------------------
# (B) Per-gate gridsynth with explicit epsilon control (recommended)
# ---------------------------------------------------------------------------
def to_clifford_t(qc: QuantumCircuit, epsilon: float = 1e-10) -> QuantumCircuit:
    """Clifford+T synthesis with a controllable approximation error ``epsilon``.

    Step 1: transpile into ``{u, cx}`` so every non-Clifford operation is a
            single-qubit gate (CX is already Clifford).
    Step 2: replace each single-qubit gate by its Ross-Selinger Clifford+T
            approximation via :func:`gridsynth_unitary`; keep every CX.
    """
    base = transpile(qc, basis_gates=["u", "cx"], optimization_level=1)
    # base = transpile(qc, basis_gates=["rz", "sx", "x", "cx"], optimization_level=1)
    print(base.draw(fold=-1))
    out = QuantumCircuit(*base.qregs)
    for inst in base.data:
        op, qubits = inst.operation, inst.qubits
        idx = [base.find_bit(q).index for q in qubits]
        if op.name == "cx":
            out.cx(*idx)                                   # Clifford entangler, kept as-is
        elif op.name == 'rz':
            approx = gridsynth_rz(op.params[0], epsilon)
            out.compose(approx, [idx[0]], inplace=True)
        elif op.num_qubits == 1 and (op.name == 'u' or op.name == 'u3'):
            approx = gridsynth_unitary(Operator(op).data, epsilon)  # 2x2 unitary -> Clifford+T
            out.compose(approx, [idx[0]], inplace=True)
        else:
            raise ValueError(f"unexpected {op.num_qubits}-qubit gate {op.name!r}")
    return out


In [50]:
qc = random_circuit(3, 10, max_operands=2, seed=12)
qc.draw(fold=-1)



┌──────────────┐                                                                          
q_0: ─────────────────────────────■─────────────────────────┤1             ├─■────────────────────────────────────X────────────────■──────────────■───
            ┌────────────┐        │U1(1.6208)               │  Rxx(5.5308) │ │ ┌──────────────────────────┐┌────┐ │                │ZZ(3.3683)    │   
q_1: ───────┤ U1(1.4485) ├────────■─────────────────■───────┤0             ├─■─┤0                         ├┤ √X ├─┼───────■────────■──────────────┼───
     ┌──────┴────────────┴──────┐    ┌───┐    ┌─────┴──────┐└──────────────┘   │  (XX-YY)(1.2651,0.84416) │└────┘ │ ┌─────┴──────┐             ┌──┴──┐
q_2: ┤ U(4.2125,0.72307,5.6317) ├────┤ I ├────┤ Rz(1.6259) ├───────────────────┤1                         ├───────X─┤ Ry(4.9033) ├─────────────┤ Sdg ├
     └──────────────────────────┘    └───┘    └────────────┘                   └──────────────────────────┘         └────────────┘             └─────┘

In [64]:
to_clifford_t(qc).count_ops()


global phase: 4.6995
     ┌─────────────┐                              ┌─────────┐  ┌────┐  ┌─────────┐                                        ┌───┐┌────────────┐┌───┐                            ┌───┐┌─────────┐ ┌────┐┌─────────┐                                                                                    ┌───┐     ┌───┐                                                                                                                                 ┌──────────┐
q_0: ┤ Rz(0.81039) ├──■─────────────────────■─────┤ Rz(π/2) ├──┤ √X ├──┤ Rz(π/2) ├────────────────────────────────────────┤ X ├┤ Rz(5.5308) ├┤ X ├────────────────────────────┤ X ├┤ Rz(π/2) ├─┤ √X ├┤ Rz(π/2) ├────────────────────────────────────────────────────────────────────────────────────┤ X ├──■──┤ X ├──────────────────────────────────────────────────────────────────────────────────────■──────────────────■────■───────────────■──┤ Rz(-π/4) ├
     └┬────────────┤┌─┴─┐ ┌──────────────┐┌─┴─┐ ┌─┴─────────┴─┐└────┘  └─────────

OrderedDict([('h', 1666), ('t', 1629), ('s', 893), ('cx', 18), ('x', 6)])

In [71]:
to_clifford_t(qc).count_ops()


global phase: 1.3689
              ┌────────────────┐                                    ┌───────────┐                                                              ┌───────────────────┐          ┌────────────────┐                                                 ┌───┐┌────────────────┐┌───┐                           ┌───┐┌────────────┐┌───┐┌─────────────┐
q_0 -> 0 ─────┤ U(0,0,0.81039) ├───────■───────────────────────■────┤ U(π,-π,0) ├────────────────────────────────────────────────────────■─────┤ U(2.3892,-π,-π/2) ├──────■───┤ U(2.4516,0,-π) ├─────────────────────────────────────────────────┤ X ├┤ U(-2.4516,0,0) ├┤ X ├───────────────────────────┤ X ├┤ U(0,0,π/4) ├┤ X ├┤ U(0,0,-π/4) ├
              ├───────────────┬┘     ┌─┴─┐┌─────────────────┐┌─┴─┐┌─┴───────────┴──┐                             ┌────────────────────┐┌─┴─┐┌──┴───────────────────┴───┐┌─┴─┐┌┴────────────────┴┐      ┌────────────────┐      ┌───────────────┐ └─┬─┘└────────────────┘└─┬─┘┌───┐┌───────────────┐┌───┐└─┬─┘└─────

OrderedDict([('h', 2191), ('t', 2142), ('s', 1133), ('x', 14), ('cx', 14)])

In [51]:
to_clifford_t_transpile(qc).count_ops()

OrderedDict([('h', 2143),
             ('t', 2010),
             ('s', 1476),
             ('sdg', 66),
             ('cx', 17),
             ('x', 6)])

In [22]:
transpile(qc, basis_gates=["rz", "sx", "x", "cx"], optimization_level=1).count_ops()

OrderedDict([('rz', 34), ('cx', 18), ('sx', 16)])

In [37]:
from qiskit.synthesis import gridsynth_rz, gridsynth_unitary
from scipy.stats import unitary_group
from qiskit.circuit.library import UGate
from qiskit.synthesis import OneQubitEulerDecomposer

euler_decomposer = OneQubitEulerDecomposer()

u = unitary_group.rvs(2)


In [48]:
params = list(euler_decomposer(u).data)[0].operation.params
print(sum([gridsynth_rz(param).count_ops()['t'] for param in params]))

311


In [47]:
gridsynth_unitary(u).count_ops()['t']

311

In [1]:
import qiskit
from qiskit import QuantumCircuit
import json
import phoenix

In [2]:
input_json = '../../benchmarks/hamlib/binaryoptimization/flat100-4-flat100-7.cnf-90-cc.json'

with open(input_json, 'r') as f:
    data = json.load(f)

def get_t_cost(qc):
    return qc.count_ops()['t'], qc.depth(lambda instr: instr.operation.name == 't')

ham = phoenix.Hamiltonian(data['paulis'], data['coeffs'])
qc = ham.generate_circuit()
print(get_t_cost(phoenix.utils.synth_to_clifford_t(qc)))

(22979, 1662)


In [4]:
qc_phoenix = phoenix.compile_hamiltonian_simulation(ham, grouping='support')
qc_symphony = phoenix.compile_hamiltonian_simulation(ham)
print('phoenix:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_phoenix)))
print('symphony:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_symphony)))

phoenix: (68437, 3675)
symphony: (70688, 3046)


In [3]:
qc_phoenix = phoenix.compile_hamiltonian_simulation(ham, grouping='support', optimize=False)
qc_symphony = phoenix.compile_hamiltonian_simulation(ham, optimize=False)
print('phoenix:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_phoenix)))
print('symphony:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_symphony)))

phoenix: (22979, 938)
symphony: (22979, 837)


In [3]:
qc_qisikt = phoenix.utils.compile_by_qiskit(ham.paulis, ham.coeffs, optimize=False)
print('qiskit:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_qisikt)))

qiskit: (22979, 1153)


In [12]:
qc_tket = phoenix.utils.compile_by_tket(ham.paulis.to_labels(), ham.coeffs, optimize=False)
print('tket:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_tket)))

tket: (22979, 729)


In [11]:
phoenix.utils.print_circ_info(phoenix.compile_hamiltonian_simulation(ham, optimize=False))

+------------+-----------+--------------+-------+----------+
| num_qubits | num_gates | num_2q_gates | depth | depth_2q |
+------------+-----------+--------------+-------+----------+
|     90     |    237    |     148      |   10  |    9     |
+------------+-----------+--------------+-------+----------+


In [8]:
phoenix.utils.print_circ_info(phoenix.utils.compile_by_tket(ham.paulis.to_labels(), ham.coeffs))


+------------+-----------+--------------+-------+----------+
| num_qubits | num_gates | num_2q_gates | depth | depth_2q |
+------------+-----------+--------------+-------+----------+
|     90     |    887    |     271      |   61  |    30    |
+------------+-----------+--------------+-------+----------+


In [23]:
input_json = '../../benchmarks/uccsd/uccsd_10.json'

with open(input_json, 'r') as f:
    data = json.load(f)

M = 500
data['paulis'] = data['paulis'][:M]
data['coeffs'] = data['coeffs'][:M]
ham = phoenix.Hamiltonian(data['paulis'], data['coeffs'])

In [24]:
qc_phoenix = phoenix.compile_hamiltonian_simulation(ham, grouping='support', optimize=False)
qc_symphony = phoenix.compile_hamiltonian_simulation(ham, optimize=False)
phoenix.utils.print_circ_info(qc_phoenix)
phoenix.utils.print_circ_info(qc_symphony)

+------------+-----------+--------------+-------+----------+
| num_qubits | num_gates | num_2q_gates | depth | depth_2q |
+------------+-----------+--------------+-------+----------+
|     10     |    1586   |     1178     |  1182 |   1000   |
+------------+-----------+--------------+-------+----------+
+------------+-----------+--------------+-------+----------+
| num_qubits | num_gates | num_2q_gates | depth | depth_2q |
+------------+-----------+--------------+-------+----------+
|     10     |    1840   |     1044     |  945  |   606    |
+------------+-----------+--------------+-------+----------+


In [25]:
print('phoenix:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_phoenix)))
print('symphony:', get_t_cost(phoenix.utils.synth_to_clifford_t(qc_symphony)))

phoenix: (46570, 27431)
symphony: (46247, 28662)


In [17]:
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Pauli

qc = QuantumCircuit(2)
qc.append(phoenix.CNOTEquivCliffordGate('Z', 'Y'), [0,1])
qc.append(PauliEvolutionGate(Pauli('ZX'), 0.1), [0, 1])
qc.append(PauliEvolutionGate(Pauli('YY'), 0.2), [0, 1])
qc.append(PauliEvolutionGate(Pauli('ZZ'), 0.3), [0, 1])
qc.append(phoenix.CNOTEquivCliffordGate('Z', 'Y'), [0,1])

qiskit.transpile(qc.decompose(), basis_gates=['u', 'cx'], optimization_level=2).draw(fold=-1)

global phase: 2.4627
       ┌───────────────┐          ┌───────────────────┐         ┌─────────────────────┐
q_0: ──┤ U(π/2,-π,π/2) ├────■─────┤ U(π,1.448,3.0188) ├──────■──┤ U(π/2,-π/2,-1.9708) ├
     ┌─┴───────────────┴─┐┌─┴─┐┌──┴───────────────────┴───┐┌─┴─┐└─┬──────────────────┬┘
q_1: ┤ U(π/2,0,-0.15207) ├┤ X ├┤ U(1.5738,1.6009,-1.7685) ├┤ X ├──┤ U(π/2,2.3229,-π) ├─
     └───────────────────┘└───┘└──────────────────────────┘└───┘  └──────────────────┘

In [16]:
qc.decompose().decompose().draw(fold=-1)

┌───┐┌───┐┌─────────┐┌───┐ ┌───┐  ┌──────┐                     ┌────┐                     
q_0: ┤ H ├┤ X ├┤ Rz(0.2) ├┤ X ├─┤ H ├──┤ √Xdg ├──■───────────────■──┤ √X ├──■───────────────■──
     └───┘└─┬─┘└─────────┘└─┬─┘┌┴───┴─┐└──────┘┌─┴─┐┌─────────┐┌─┴─┐├────┤┌─┴─┐┌─────────┐┌─┴─┐
q_1: ───────■───────────────■──┤ √Xdg ├────────┤ X ├┤ Rz(0.4) ├┤ X ├┤ √X ├┤ X ├┤ Rz(0.6) ├┤ X ├
                               └──────┘        └───┘└─────────┘└───┘└────┘└───┘└─────────┘└───┘

In [ ]:


phoenix.primitive.utils._synthesize_successive_2q_pauli_rotation_block(qc)

In [10]:
qc_symphony.count_ops()

OrderedDict([('rzz', 134), ('rz', 89), ('cxz', 14)])

In [10]:
qc_phoenix.count_ops()

OrderedDict([('u', 804), ('cx', 268), ('rz', 89), ('cxz', 14)])